# 02 Human in the Loop
In this notebook we'll learn how to add supervision to our Agent with a "Human in the Loop". Where should we put the Human? After all the goal of the agent is to automate. So where to (or even if we should) put the human is just as important a decision as the tools to give our agent, or the model we choose to pair with our agent.

We can boil down the many use-cases for "Human in the Loop" to 3 main categories:
1. **Approving sensitive actions:** such as making claims _disbursements_ as determined by a Claims Agent.
2. **Adding missing content** - such as making additions (e.g. add venue for conference) for an conference planning email generated by agent. 
3. **Debugging our Agent** - 

In this notebook we'll cover the 1st category "Approving Sensitive Actions". The techniques we discuss here are the exact same techniques you'd use for the two other types of use-cases.

Let's say I have an Email inbox assistant, which can read email in my in-box and can summarize emails for me - nothing too risky about this so far. But let's say it can also draft and send emails on my behalf - this is maybe an action whare you'd like to have the final say (before the email is actually sent). The example below does just that.

Here are the steps our example below tries to simulate:
1. Assume some function (not included in this example) has read email from my inbox
2. This email (body of email) is passed into the Agent via the AgentState in the `invoke()` call
3. The agent first calls `read_email` tool to get contents of email (from AgentState or Runtime)
4. Then it does what user asks it to do. For example, "send an appropriate reply"
5. To _send_ email the `send_email` tool is called, which is _interrupted_ by the HTIL middleware

In [2]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

Below are _dummy_ tool functions to read & send email.

In [ ]:
from langchain.tools import tool, ToolRuntime


@tool
def read_email(runtime: ToolRuntime) -> str:
    """read email from AgentState for model's consumption"""
    return runtime.state["email"]


@tool
def send_email(body: str) -> str:
    """sends the email to a given address with a given subject & email body"""
    # in production (actual agent), you'll put the code to send
    # email here
    return f"Email ------ \n {body}\n\nEmail Sent!"

When defining our agent, we use LangChain's pre-defined middleware `HumanInTheLoopMiddleware`. Here is an example of how to use it in the `create_agent()` call:

```python
import...  # other imports
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    model="openai:gpt-5-nano",
    ...,
    tools=[tool1, tool2, tool3],   # assume these functions are defined elsewhere
    # here is our HTIL middleware
    middleware=[
        HumanInTheLoopMiddleware(
            # here we list all our tools and tell the middleware
            # on which tool to "fire" the HTIL middleware
            interrupt_on={
                # key = name of tool function & value = True (fire) or False (ignore)
                "tool1": False,
                # will fire HTIL AFTER tool3 call returns to agent
                "tool2": True,
                "tool3": False,
            },
            description_prefix="Tool execution requires approval",
        )
    ],
)
```

With our `read_email` and `send_email` tools defined above, we don't need HTIL for `read_mail`, but we _certainly do_ for `send_email`. Here is how we'll define our agent.

In [ ]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware


class EmailState(AgentState):
    email: str


agent = create_agent(
    model="openai:gpt-5-nano",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                # send_email tool required Human Approval
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        )
    ],
)

In [4]:
from langchain.messages import HumanMessage
from pprint import pprint

config = {"configurable": {"thread_id": "htil_id1"}}

response = agent.invoke(
    {
        "messages": [
            HumanMessage("Please read the email & send an appropriate response")
        ],
        "email": "Hi Bilbo, I'm gonna be late for our meeting tomorrow. Can we re-schedule? Best, Frodo",
    },
    config=config,
)

When I print the contents of the response, notice that the very first message has the `__interrupt__` key

In [14]:
pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Subject: '
                                                                          'Re: '
                                                                          'Meeting '
                                                                          'tomorrow\n'
                                                                          '\n'
                                                                          'Hi '
                                                                          'Frodo,\n'
                                                                          '\n'
                                                                          'No '
                                                                          'problem—thanks '
                                                                          'for '
                                                                          'the '
    

In [18]:
response["messages"][-1]

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1054, 'prompt_tokens': 205, 'total_tokens': 1259, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 960, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DXoZddr3Ts6tiLoKzmLV7iTfsvwPl', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dba91-a5a2-7de2-8dc6-89c8200cf592-0', tool_calls=[{'name': 'send_email', 'args': {'body': 'Subject: Re: Meeting tomorrow\n\nHi Frodo,\n\nNo problem—thanks for the heads-up. We can reschedule. What time tomorrow works for you? I’m available at 10:00 AM or 2:00 PM, or let me know a time that’s convenient and I’ll adapt.\n\nBest,\nBilbo'}, 'id': 'call_C9yHPic4j1XXupQuMPHO8DyS', 'type': 'tool_

In [6]:
# and here is the generated email body
print(response["messages"][-1].tool_calls[0]["args"]["body"])

Hi Frodo,

No problem—thanks for the heads up. I can reschedule. Would 11:00 AM or 2:00 PM tomorrow work for you? If neither works, tell me a time that does and I’ll adjust.

Best,
Bilbo


Now let's write a **utility function** that processes the "HTIL-enabled" request in a loop. When "_interrupt_"-ed, it will display the genarated email to the user with a `(y/n)` prompt asking if we should send email. If user responds with a `y`, it will send the email, if user responds with a `n`, it will loop back and re-generate an email until user responds with a `y` or `quit` to abort process.

In [7]:
def process_email_with_htil(email_body: str):
    """Invoke agent to reply to email_body, looping on HTIL until user approves or quits."""
    thread_id = f"htil_{uuid.uuid4().hex[:8]}"
    config = {"configurable": {"thread_id": thread_id}}
    human_message = "Please read the email & send an appropriate response"

    while True:
        response = agent.invoke(
            {
                "messages": [HumanMessage(human_message)],
                "email": email_body,
            },
            config=config,
        )

        if "__interrupt__" not in response:
            print("Agent completed without interruption.")
            return response

        # Extract the email body the agent wants to send
        last_msg = response["messages"][-1]
        try:
            generated_body = last_msg.tool_calls[0]["args"]["body"]
        except (AttributeError, IndexError, KeyError):
            print("Could not extract generated email body from agent response.")
            return response

        print(f"\nGenerated email reply:\n{'-' * 40}\n{generated_body}\n{'-' * 40}\n")

        while True:
            choice = (
                input("I have generated this email, proceed to send (y/n)? ")
                .strip()
                .lower()
            )

            if choice == "y":
                # Resume the agent — it will execute the pending send_email tool call
                final_response = agent.invoke(None, config=config)
                print("\nEmail sent successfully!")
                return final_response

            elif choice == "n":
                user_input = input(
                    "Enter modified email body (or 'quit' to exit):\n> "
                ).strip()

                if user_input.lower() == "quit":
                    print("Operation cancelled.")
                    return None

                # Re-invoke agent on a fresh thread, instructing it to send the user's text
                thread_id = f"htil_{uuid.uuid4().hex[:8]}"
                config = {"configurable": {"thread_id": thread_id}}
                human_message = f"Please send this exact email reply:\n{user_input}"
                break  # restart outer loop with new message

            else:
                print("Please enter 'y' or 'n'.")

In [ ]:
incoming_email = (
    "Hi Bilbo, I'm gonna be late for our meeting tomorrow. "
    "Can we re-schedule? Best, Frodo"
)
result = process_email_with_htil(incoming_email)
if result:
    pprint(result)